# DiagramIO: đo hiệu năng pha 2

Notebook chạy bộ đo trong thư mục `bench/` của repo DiagramIO (xem `bench/README.md` và `docs/adr/0001-phuong-phap-do.md`).

- **Settings**: Accelerator **None** (CPU), bật **Internet**.
- Chạy nền để không phụ thuộc trình duyệt: **Save Version → Save & Run All (Commit)**. Ma trận `m0` mất khoảng 3 giờ, `quick` khoảng 10 phút (nên chạy `quick` trước).
- Kết quả: thư mục `DiagramIO/bench/results/<phiên>/` và file `results.tgz` trong tab **Output**. Giải nén vào `bench/results/` của repo rồi commit các file tóm tắt.
- So sánh hai cấu hình (từ M1): đặt thêm `REF_C1` và dùng ma trận tương ứng; hai cấu hình chạy xen kẽ trong cùng phiên.

In [ ]:
REPO_URL = "https://github.com/Phamminh161005/DiagramIO.git"
REF = "main"      # commit hoặc tag có thư mục bench/; cấu hình C0 được build từ REF
MATRIX = "m0"     # "quick" để chạy thử vài phút với dữ liệu nhỏ

In [ ]:
import os
import subprocess

os.chdir("/kaggle/working")
if not os.path.isdir("DiagramIO"):
    subprocess.run(["git", "clone", REPO_URL, "DiagramIO"], check=True)
os.chdir("/kaggle/working/DiagramIO")
subprocess.run(["git", "fetch", "--tags", "origin"], check=True)
subprocess.run(["git", "checkout", "--detach", REF], check=True)
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

In [ ]:
# Cài PostgreSQL 16, JDK 21, k6, async-profiler và cấu hình PostgreSQL (khoảng 2-3 phút)
!bash bench/kaggle/setup.sh

In [ ]:
# Build jar, sinh và nạp dữ liệu, chạy cả ma trận đo, tổng hợp báo cáo
!REF_C0={REF} bash bench/kaggle/run_matrix.sh {MATRIX}

In [ ]:
# Đóng gói toàn bộ kết quả (kể cả log thô) để tải về từ tab Output
!cd bench/results && tar czf /kaggle/working/results.tgz . && ls -lh /kaggle/working/results.tgz

In [ ]:
import glob
from IPython.display import Markdown, display

latest = sorted(glob.glob("bench/results/*/report.md"))[-1]
print(latest)
display(Markdown(open(latest, encoding="utf-8").read()))